In [ ]:
Date: July 4, 2026 Saturday

# 算法笔记：二叉树子树匹配的双重递归（DFS）解法

## 1. Algorithm Pattern (算法范式)
* **Double Recursion (双重递归)**：外层递归负责遍历主树 `root` 的每一个节点，内层递归（辅助函数）负责以当前节点为根，与目标子树 `subRoot` 进行结构与数值的完全同构比对。
* **Depth-First Search (DFS)**：自顶向下深入探索树的每一个分支，属于典型的暴力匹配机制。

## 2. 核心思想与复杂度分析
本解法通过朴素的拓扑结构比对来验证子树关系。
* **时间复杂度**：最坏情况下为 $O(M \times N)$（其中 $M$ 和 $N$ 分别为主树和子树的节点数）。当两棵树高度不平衡且存在大量高度相似的局部结构时，算法会引发频繁的深层回溯，导致性能退化。
* **空间复杂度**：$O(\max(H_{root}, H_{subRoot}))$，系统开销取决于递归调用栈的深度（即树的高度 $H$）。

---

## 3. 工程设计中的关键边界控制

双重递归的优雅之处在于对 `None`（空节点）的严密条件分流。外层与内层函数各自承担不同的边界守护：

### A. 内层同构比对边界 (`sameTree`)
验证两棵树是否完全相同，必须通过三层逻辑递进：
1. **全空复合**：若 `p` 和 `q` 同时为 `None`，说明两树同时到达叶子边界，结构一致，返回 `True`。
2. **单侧落空/值异构**：若仅有一侧为 `None`，或节点数值不相等（`p.val != q.val`），说明拓扑或语义已被打破，立刻熔断返回 `False`。
3. **分治递归**：若当前节点完全一致，则同步校验它们的左子树与右子树。

### B. 外层遍历边界 (`isSubtree`)
1. **主树穷尽**：若主树已被搜索至空（`not root`），说明未找到匹配子树，返回 `False`。
2. **即时命中**：优先调用 `sameTree(root, subRoot)`。若当前位置吻合，则无需继续向下探索。
3. **分流检索**：若当前位置不匹配，利用逻辑或（`or`）的短路特性，递归检索主树的左子树和右子树。

---

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isSubtree(self, root: Optional[TreeNode], subRoot: Optional[TreeNode]) -> bool:
        
        # Create a helper function to traversal both trees

        def sameTree(p, q):

            # Tree p & q are both empty ——> True
            if (not p) and (not q):
                return True

            # Tree p & q one of them empty ——> False
            # Tree p & q are not equal ——> False
            if (not p or not q) or (p.val != q.val):
                return False       
        
            return sameTree(p.left, q.left) and sameTree(p.right, q.right)

        # Check if there's a subtree
        # Base Case:
        if not root: return False

        # Call the helper() to check if there's a subtree
        if sameTree(root, subRoot):
            return True

        # If the current root not equal, recursively check the subtrees
        return self.isSubtree(root.left, subRoot) or self.isSubtree(root.right, subRoot)
        

# 算法笔记：二叉树子树匹配的序列化（降维）解法

## 1. Algorithm Pattern (算法范式)
* **Dimensionality Reduction (降维映射)**：将复杂的二维非线性数据结构（树），通过带有严格占位符的前序遍历，无损压缩为一维的线性数据结构（字符串）。
* **Pattern Matching (模式匹配)**：将“树的同构比对”转化为底层的“字符串子串检索”，从而直接调用系统级优化的高效匹配算法。

## 2. 核心思想与复杂度优势
传统的深度优先搜索（DFS）暴力匹配存在大量冗余回溯，最坏时间复杂度为 $O(M \times N)$。
本解法通过序列化降维，将时间复杂度严格控制在 $O(M + N)$，空间复杂度为 $O(M + N)$。

---

## 3. 工程设计中的关键防错逻辑

在将树状结构展平为线性字符串的过程中，极易丢失拓扑信息或产生语义歧义。本算法通过两项严密的约束来保证数据的**唯一映射（双射）**：

### A. 结构信息防丢失：引入空节点占位符 `#`
“Null/None” 在系统设计中是界定数据边界的重要信息。如果不记录空节点，会导致拓扑结构不同的树生成完全相同的字符串。
* **正确约束**：引入 `#` 后，左子树（根 `1`，左 `2`，右空）变为 `"12###"`；右子树（根 `1`，左空，右 `2`）变为 `"1#2##"`，结构被精准保留。

### B. 语义粘连防范：引入边界隔离符 `^` 与 `$`
在拼接数值时，必须为每一个独立的数据单元打上强边界约束，以防止**边界溢出（Boundary Bleed）**。

| 节点真实值 | 裸拼序列化 (存在漏洞) | 强边界序列化 (严密设计) | 逻辑分析 |
| :--- | :--- | :--- | :--- |
| 主树节点 `12` | `"12##"` | `"^12$##"` | 若无隔离，字符串 `"12##"` 会错误地包含 `"2##"`。 |
| 子树节点 `2` | `"2##"` | `"^2$##"` | 增加 `^` 和 `$` 后，`"^12$##"` 绝不会匹配 `"^2$##"`，彻底封死歧义。 |

---



In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isSubtree(self, root: Optional[TreeNode], subRoot: Optional[TreeNode]) -> bool:
        def serialize(node):
            if not node:
                return "#"
            # 严格使用 ^ 和 $ 包裹节点值，构建强边界约束
            return f"^{node.val}${serialize(node.left)}{serialize(node.right)}"
        
        # 将二维树结构拍平为一维字符串基因链
        root_str = serialize(root)
        subRoot_str = serialize(subRoot)
        
        # 利用底层优化的字符串模式匹配完成检索
        return subRoot_str in root_str
    
    # Time Complexity: O(n + m), where n is the number of nodes in the root tree and m is the number of nodes in the subRoot tree. The serialization of both trees takes O(n) and O(m) time respectively, and checking if subRoot_str is a substring of root_str takes O(n + m) time in the worst case.